## `data_loader.py`

### Module Purpose

This notebook is a **documented, executable mirror** of `src/data_loader.py`.
It exposes the three public helpers that every other notebook and pipeline stage relies on:

| Function | Responsibility |
|---|---|
| `load_data()` | Read the raw CSV from the path defined in `config` |
| `encode_owner()` | Ordinal-encode the `owner` column via `config.owner_mapping` |
| `check_structure()` | Print a quick sanity-check summary of any DataFrame |


In [1]:
import pandas as pd
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'src'))

import config


Reads the raw CSV and returns a DataFrame with **no transformations applied**.
Keeping loading and transformation separate means you can always inspect
the original data before any mutation occurs.


In [2]:
def load_data(path=config.raw_data_path):
    df = pd.read_csv(path)
    return df

#### `encode_owner()`

Converts the free-text `owner` column into an **ordered integer column**
using `config.owner_mapping`.

```


In [3]:
def encode_owner(df: pd.DataFrame):
    df = df.copy()
    before_dtype = df["owner"].dtype
    df["owner"] = df["owner"].map(config.owner_mapping)

    unmapped = df["owner"].isna().sum()
    if unmapped:
        print(
            f"warning: {unmapped} row(s) had an Owner value not found in "
            f"owner_mapping and became NaN. Check df['owner'].unique() "
            f"against config.owner_mapping."
        )
    print(
        f"Encoded 'owner': {before_dtype} -> {df['owner'].dtype} "
        f"using owner_mapping."
    )
    return df

#### `check_structure()`

Prints a concise structural summary of any DataFrame.
It is intentionally **read-only** — it never mutates `df`.


In [4]:
def check_structure(df: pd.DataFrame):
    print("\nStructure check:")
    print(f"\nnumber of rows {df.shape[0]}, number of columns {df.shape[1]}")
    print(f"\nColumn dtypes")
    print(f"{df.dtypes}")
    print("\nfirst 5 rows:")
    print(f"{df.head()}")
    print("\nmissing values:")
    print(f"{df.isna().sum()}")
    print("\nduplicate rows")
    print(f"{df.duplicated().sum()}")

### What each check reveals

| Print statement | What it tells you |
|---|---|
| `df.shape` | Whether the expected number of rows and columns loaded correctly |
| `df.dtypes` | Whether pandas inferred sensible types (e.g. `float64` for price, `object` for strings) |
| `df.head()` | A visual sanity-check that columns are aligned and values look plausible |
| `df.isna().sum()` | Which columns have missing data and how many rows are affected — critical before any model training |
| `df.duplicated().sum()` | Whether identical rows exist, which would silently bias model evaluation if not removed |


## Quick Test Run

Execute the full pipeline — load, encode, check — to verify everything works end-to-end.


In [5]:

df_raw = load_data()

df_encoded = encode_owner(df_raw)

check_structure(df_encoded)

Encoded 'owner': str -> float64 using owner_mapping.

Structure check:

number of rows 4340, number of columns 8

Column dtypes
name                 str
year             float64
selling_price      int64
km_driven        float64
fuel                 str
seller_type          str
transmission         str
owner            float64
dtype: object

first 5 rows:
                       name    year  selling_price  km_driven    fuel  \
0             Maruti 800 AC  2007.0          60000    70000.0  Petrol   
1  Maruti Wagon R LXI Minor  2007.0         135000    50000.0  Petrol   
2      Hyundai Verna 1.6 SX  2012.0         600000   100000.0  Diesel   
3    Datsun RediGO T Option  2017.0         250000    46000.0  Petrol   
4     Honda Amaze VX i-DTEC  2014.0         450000   141000.0  Diesel   

  seller_type transmission  owner  
0  Individual       Manual    1.0  
1  Individual       Manual    NaN  
2  Individual       Manual    1.0  
3  Individual       Manual    1.0  
4  Individual       Manu